In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Model Architecture: LSTM for Sentiment Classification

We will implement a **many-to-one LSTM architecture** from scratch using NumPy for binary sentiment classification on the IMDb dataset.

### Architecture

1. **Input:** Movie reviews from the IMDb dataset.
2. **Tokenisation:** Convert reviews into sequences of integer token IDs.
3. **Padding and Truncation:** Standardise each review to a maximum length of 100 tokens.
4. **Embedding Layer:** Trainable embedding matrix that converts each token ID into a 32-dimensional vector.
5. **LSTM Layer:** A single LSTM layer with 64 hidden units, processing the input sequence sequentially.
6. **Dense Layer:** Maps the final hidden state (h_T) to 2 output logits.
7. **Softmax:** Converts logits into probabilities for the two sentiment classes:
   - 0 — Negative
   - 1 — Positive

### Initial Hyperparameters

| Parameter | Value |
|---|---:|
| Vocabulary size | 10,000 |
| Embedding dimension | 32 |
| Maximum sequence length | 100 |
| Number of LSTM layers | 1 |
| Hidden dimension | 64 |
| Output classes | 2 |
| Batch size | 32 |

### Implementation Approach

- NumPy-based implementation from scratch.
- Manual forward propagation and Backpropagation Through Time (BPTT).
- Trainable word embeddings and LSTM parameters.
- Model selection based on validation loss.
- Save the best model checkpoint, including learned parameters and configuration, for future reuse.

**Architecture flow:**

`Review → Tokenisation → Embedding → LSTM → Dense → Softmax → Sentiment Prediction`

In [4]:
print(df.shape)
print(df.columns)

(50000, 2)
Index(['review', 'sentiment'], dtype='str')


In [5]:
df.sentiment=df.sentiment.map({"negative":0,"positive":1})
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [7]:
df["review"] = df["review"].str.lower()

In [9]:
df["tokens"] = df["review"].apply(lambda x: x.split())

In [10]:
print("Original review:\n", df["review"].iloc[0][:300])
print("\nTokenised review:\n", df["tokens"].iloc[0][:30])
print("\nNumber of tokens:", len(df["tokens"].iloc[0]))

Original review:
 one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. tru

Tokenised review:
 ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', "you'll", 'be', 'hooked.', 'they', 'are', 'right,', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me.<br', '/><br', '/>the']

Number of tokens: 307


In [6]:
from sklearn.model_selection import train_test_split

In [12]:
df.head()

,review,sentiment,tokens
0,one of the other reviewers has mentioned that ...,1,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production. <br /><br />the...,1,"[a, wonderful, little, production., <br, /><br..."
2,i thought this was a wonderful way to spend ti...,1,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there's a family where a little boy ...,0,"[basically, there's, a, family, where, a, litt..."
4,"petter mattei's ""love in the time of money"" is...",1,"[petter, mattei's, ""love, in, the, time, of, m..."


In [11]:
X = df["tokens"]
y = df["sentiment"]

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y,test_size=0.30,random_state=42,stratify=y)

In [14]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp,test_size=0.50,random_state=42,stratify=y_temp)

In [16]:
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

Training samples: 35000
Validation samples: 7500
Testing samples: 7500


In [17]:
word_counts = {}

for review in X_train:
    for word in review:
        if word in word_counts:
            word_counts[word] += 1
        else:
            word_counts[word] = 1

print("Unique words in training data:", len(word_counts))

Unique words in training data: 311486


In [18]:
vocab_size = 10000

sorted_words = sorted(word_counts.items(),key=lambda x: x[1],reverse=True)

word_to_id = {"<PAD>": 0,"<OOV>": 1}

for idx, (word, count) in enumerate(sorted_words[:vocab_size - 2], start=2):
    word_to_id[word] = idx

print("Final vocabulary size:", len(word_to_id))

Final vocabulary size: 10000


In [24]:
sorted_words[:5]

[('the', 445582),
 ('a', 221116),
 ('and', 218872),
 ('of', 199975),
 ('to', 184684)]

In [25]:
word_to_id

{'<PAD>': 0,
 '<OOV>': 1,
 'the': 2,
 'a': 3,
 'and': 4,
 'of': 5,
 'to': 6,
 'is': 7,
 'in': 8,
 'i': 9,
 'this': 10,
 'that': 11,
 'it': 12,
 '/><br': 13,
 'was': 14,
 'as': 15,
 'for': 16,
 'with': 17,
 'but': 18,
 'on': 19,
 'movie': 20,
 'are': 21,
 'his': 22,
 'not': 23,
 'you': 24,
 'film': 25,
 'have': 26,
 'be': 27,
 'he': 28,
 'one': 29,
 'at': 30,
 'by': 31,
 'an': 32,
 'they': 33,
 'from': 34,
 'all': 35,
 'who': 36,
 'like': 37,
 'so': 38,
 'just': 39,
 'or': 40,
 'has': 41,
 'about': 42,
 'her': 43,
 "it's": 44,
 'if': 45,
 'some': 46,
 'out': 47,
 'what': 48,
 'when': 49,
 'very': 50,
 'there': 51,
 'more': 52,
 'would': 53,
 'even': 54,
 'good': 55,
 'she': 56,
 'my': 57,
 'their': 58,
 'no': 59,
 'only': 60,
 'really': 61,
 'had': 62,
 'which': 63,
 'up': 64,
 'can': 65,
 'see': 66,
 'were': 67,
 'than': 68,
 'we': 69,
 '-': 70,
 'been': 71,
 'get': 72,
 'will': 73,
 'into': 74,
 'much': 75,
 'because': 76,
 'story': 77,
 'most': 78,
 'how': 79,
 'other': 80,
 "don't":

In [26]:
def encode_review(review, word_to_id):
    return [word_to_id.get(word, 1) for word in review]

In [27]:
X_train_encoded = [encode_review(review, word_to_id) for review in X_train]
X_val_encoded = [encode_review(review, word_to_id) for review in X_val]
X_test_encoded = [encode_review(review, word_to_id) for review in X_test]

In [28]:
print("Original review:", X_train.iloc[0][:15])
print("Encoded review:", X_train_encoded[0][:15])

print("\nEncoded review length:", len(X_train_encoded[0]))

Original review: ['if', 'you', 'want', 'to', 'see', 'a', 'true', 'thriller,', 'rent', 'this!!!', "it's", 'not', 'from', 'the', 'director']
Encoded review: [45, 24, 172, 6, 66, 3, 311, 3407, 817, 1, 44, 23, 34, 2, 205]

Encoded review length: 53


In [37]:
max_length = 100
pad_id = 0

def pad_sequence(sequence, max_length):
    if len(sequence) < max_length:
        sequence = sequence + [pad_id] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]

    return sequence


In [38]:
X_train_padded = np.array([
    pad_sequence(review, max_length) for review in X_train_encoded
])

X_val_padded = np.array([
    pad_sequence(review, max_length) for review in X_val_encoded
])

X_test_padded = np.array([
    pad_sequence(review, max_length) for review in X_test_encoded
])


In [40]:
print("Training shape:", X_train_padded.shape)
print("Validation shape:", X_val_padded.shape)
print("Testing shape:", X_test_padded.shape)

Training shape: (35000, 100)
Validation shape: (7500, 100)
Testing shape: (7500, 100)


In [49]:
import random

num = random.random()


0.34818157217150136
